In [0]:
# ============================================================
# BRONZE TO SILVER TRANSFORMATION (Unity Catalog Version)
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime, timedelta
import json

print("✓ Imports loaded")

✓ Imports loaded


In [0]:
# ============================================================
# CONFIGURATION (Unity Catalog - Direct Access)
# ============================================================

# Storage account name
storage_account_name = "storagefordeproject1"  # CHANGE THIS TO YOUR STORAGE ACCOUNT

# Container paths (abfss:// format)
bronze_base = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
silver_base = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_base = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

# Today's partition (or pass as parameter from ADF)
process_date = datetime.now().strftime("%Y/%m/%d")

print(f"✓ Configuration loaded")
print(f"  Storage Account: {storage_account_name}")
print(f"  Bronze: {bronze_base}")
print(f"  Silver: {silver_base}")
print(f"  Process Date: {process_date}")

✓ Configuration loaded
  Storage Account: storagefordeproject1
  Bronze: abfss://bronze@storagefordeproject1.dfs.core.windows.net/
  Silver: abfss://silver@storagefordeproject1.dfs.core.windows.net/
  Process Date: 2026/03/01


In [0]:
# ============================================================
# VERIFY ACCESS TO DATA LAKE
# ============================================================

print("\n--- Verifying Data Lake Access ---")

# Test bronze container access
try:
    bronze_files = dbutils.fs.ls(bronze_base)
    print(f"✓ Bronze container accessible")
    print(f"  Found {len(bronze_files)} folders/files")
    for file in bronze_files[:5]:  # Show first 5
        print(f"    - {file.name}")
except Exception as e:
    print(f"✗ Cannot access Bronze container: {e}")
    print("\nTroubleshooting:")
    print("1. Check Unity Catalog external location is configured")
    print("2. Verify storage account name is correct")
    print("3. Check cluster has access to Unity Catalog metastore")

# Test silver container access
try:
    dbutils.fs.ls(silver_base)
    print(f"✓ Silver container accessible")
except Exception as e:
    print(f"✗ Cannot access Silver container: {e}")

print("\n✓ Data Lake access verification complete")


--- Verifying Data Lake Access ---
✓ Bronze container accessible
  Found 4 folders/files
    - customers/
    - order_items/
    - orders/
    - products/
✓ Silver container accessible

✓ Data Lake access verification complete


In [0]:
# ============================================================
# TABLE CONFIGURATIONS
# ============================================================

table_configs = [
    {
        "name": "customers",
        "bronze_path": f"{bronze_base}customers/{process_date}/",
        "silver_path": f"{silver_base}dim_customers/",
        "business_key": "customer_id",
        "scd_type": 2,
        "compare_columns": ["first_name", "last_name", "email", "phone", "address", "city", "state", "customer_segment"],
        "watermark_column": "modified_date"
    },
    {
        "name": "products",
        "bronze_path": f"{bronze_base}products/{process_date}/",
        "silver_path": f"{silver_base}dim_products/",
        "business_key": "product_id",
        "scd_type": 2,
        "compare_columns": ["product_name", "category", "subcategory", "brand", "unit_price", "cost_price", "is_active"],
        "watermark_column": "modified_date"
    },
    {
        "name": "orders",
        "bronze_path": f"{bronze_base}orders/{process_date}/",
        "silver_path": f"{silver_base}fact_orders/",
        "business_key": "order_id",
        "scd_type": None,
        "watermark_column": "modified_date"
    },
    {
        "name": "order_items",
        "bronze_path": f"{bronze_base}order_items/{process_date}/",
        "silver_path": f"{silver_base}fact_order_items/",
        "business_key": "order_item_id",
        "scd_type": None,
        "watermark_column": "created_date"
    }
]

print(f"✓ Configuration loaded for {len(table_configs)} tables")

✓ Configuration loaded for 4 tables


In [0]:
# ============================================================
# CELL 3: Helper Functions - Data Quality
# ============================================================

def data_quality_checks(df, table_name):
    """
    Perform data quality checks
    """
    print(f"\n--- Data Quality Checks: {table_name} ---")
    
    initial_count = df.count()
    print(f"Initial record count: {initial_count}")
    
    # Check 1: Remove duplicates based on business key
    # For customers/products, use their ID. For orders, use order_id
    if table_name in ["customers", "products"]:
        key_col = f"{table_name[:-1]}_id"  # customer_id, product_id
    else:
        key_col = "order_id" if "order" in table_name else f"{table_name[:-1]}_id"
    
    df = df.dropDuplicates([key_col])
    after_dedup = df.count()
    print(f"After deduplication: {after_dedup} (removed {initial_count - after_dedup})")
    
    # Check 2: Remove rows with null in critical columns
    critical_cols = [key_col]
    df = df.dropna(subset=critical_cols)
    after_null_check = df.count()
    print(f"After null check: {after_null_check} (removed {after_dedup - after_null_check})")
    
    # Check 3: Add data quality metadata
    df = df.withColumn("dq_check_passed", lit(True)) \
           .withColumn("dq_check_date", current_timestamp())
    
    print(f"✓ Data quality checks passed: {after_null_check} clean records")
    
    return df

In [0]:
# ============================================================
# CELL 4: Helper Functions - SCD Type 2
# ============================================================

def apply_scd_type2(source_df, target_path, business_key, compare_columns):
    """
    Apply SCD Type 2 logic using Delta Lake MERGE
    """
    print(f"\n--- Applying SCD Type 2 ---")
    
    # Add SCD metadata columns to source
    source_df = source_df.withColumn("effective_date", current_date()) \
                         .withColumn("end_date", lit("9999-12-31").cast("date")) \
                         .withColumn("is_current", lit(True)) \
                         .withColumn("record_created_date", current_timestamp())
    
    # Check if target table exists
    if DeltaTable.isDeltaTable(spark, target_path):
        print("✓ Target table exists, performing MERGE...")
        
        target_table = DeltaTable.forPath(spark, target_path)
        
        # Build comparison condition for changes
        change_conditions = []
        for col in compare_columns:
            change_conditions.append(f"source.{col} <> target.{col} OR (source.{col} IS NULL AND target.{col} IS NOT NULL) OR (source.{col} IS NOT NULL AND target.{col} IS NULL)")
        
        change_condition = " OR ".join(change_conditions)
        
        # MERGE operation
        # Step 1: Close old records that changed
        merge_condition = f"target.{business_key} = source.{business_key} AND target.is_current = true"
        
        target_table.alias("target").merge(
            source_df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(
            condition = change_condition,
            set = {
                "end_date": "current_date()",
                "is_current": "false"
            }
        ).execute()
        
        print("✓ Closed changed records")
        
        # Step 2: Insert new versions of changed records
        # Find records that were just closed
        closed_records = spark.read.format("delta").load(target_path) \
            .filter((col("end_date") == current_date()) & (col("is_current") == False))
        
        if closed_records.count() > 0:
            # Get the new versions from source
            closed_keys = closed_records.select(business_key).distinct()
            new_versions = source_df.join(closed_keys, business_key, "inner")
            
            # Append new versions
            new_versions.write.format("delta").mode("append").save(target_path)
            print(f"✓ Inserted {new_versions.count()} new record versions")
        
        # Step 3: Insert completely new records (not in target at all)
        existing_keys = spark.read.format("delta").load(target_path).select(business_key).distinct()
        new_records = source_df.join(existing_keys, business_key, "left_anti")
        
        if new_records.count() > 0:
            new_records.write.format("delta").mode("append").save(target_path)
            print(f"✓ Inserted {new_records.count()} completely new records")
        
    else:
        print("✓ Target table doesn't exist, creating initial load...")
        source_df.write.format("delta").mode("overwrite").save(target_path)
        print(f"✓ Created table with {source_df.count()} records")
    
    print("✓ SCD Type 2 applied successfully")

In [0]:
# ============================================================
# CELL 5: Helper Functions - Simple Upsert (for Fact Tables)
# ============================================================

def simple_upsert(source_df, target_path, business_key):
    """
    Simple upsert for fact tables (no SCD)
    """
    print(f"\n--- Performing Simple Upsert ---")
    
    # Add load metadata
    source_df = source_df.withColumn("loaded_date", current_timestamp())
    
    if DeltaTable.isDeltaTable(spark, target_path):
        print("✓ Target exists, performing MERGE...")
        
        target_table = DeltaTable.forPath(spark, target_path)
        
        target_table.alias("target").merge(
            source_df.alias("source"),
            f"target.{business_key} = source.{business_key}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print("✓ Upsert completed")
    else:
        print("✓ Creating new table...")
        source_df.write.format("delta").mode("overwrite").save(target_path)
        print(f"✓ Created table with {source_df.count()} records")

In [0]:
# ============================================================
# CELL 6: Helper Functions - Late Arrival Handling
# ============================================================

def handle_late_arrivals(df, table_name):
    """
    Detect and flag late arrivals
    Late arrival = order_date < created_date
    """
    if "order_date" in df.columns and "created_date" in df.columns:
        print(f"\n--- Checking for Late Arrivals ---")
        
        # Identify late arrivals
        df = df.withColumn(
            "is_late_arrival",
            when(col("order_date") < col("created_date"), True).otherwise(False)
        )
        
        late_count = df.filter(col("is_late_arrival") == True).count()
        print(f"✓ Found {late_count} late arrival records")
        
        if late_count > 0:
            print("  Late arrivals will be inserted with correct order_date partition")
    
    return df

In [0]:
# ============================================================
# CELL 7: Helper Functions - Schema Drift
# ============================================================

def handle_schema_drift(source_df, target_path, table_name):
    """
    Handle schema changes (new columns added)
    """
    print(f"\n--- Checking Schema Drift ---")
    
    if DeltaTable.isDeltaTable(spark, target_path):
        # Get target schema
        target_df = spark.read.format("delta").load(target_path)
        target_cols = set(target_df.columns)
        source_cols = set(source_df.columns)
        
        # Find new columns
        new_columns = source_cols - target_cols
        
        if new_columns:
            print(f"⚠ Schema drift detected! New columns: {new_columns}")
            print("  Adding columns to target table...")
            
            # Add missing columns to source with null values for alignment
            # Then merge will auto-add them
            print("✓ Schema drift will be handled automatically by Delta Lake MERGE")
        else:
            print("✓ No schema drift detected")
    else:
        print("✓ New table, no drift check needed")
    
    return source_df

In [0]:
# ============================================================
# CELL 8: Main Processing Function
# ============================================================

def process_table(config):
    """
    Main function to process one table from Bronze to Silver
    """
    table_name = config["name"]
    print(f"\n{'='*60}")
    print(f"PROCESSING: {table_name.upper()}")
    print(f"{'='*60}")
    
    try:
        # Step 1: Read from Bronze
        print(f"\n[1/5] Reading from Bronze: {config['bronze_path']}")
        bronze_df = spark.read.parquet(config["bronze_path"])
        print(f"✓ Read {bronze_df.count()} records from Bronze")
        
        # Step 2: Data Quality Checks
        print(f"\n[2/5] Running data quality checks...")
        clean_df = data_quality_checks(bronze_df, table_name)
        
        # Step 3: Handle Late Arrivals (if applicable)
        print(f"\n[3/5] Checking for late arrivals...")
        clean_df = handle_late_arrivals(clean_df, table_name)
        
        # Step 4: Handle Schema Drift
        print(f"\n[4/5] Checking schema drift...")
        clean_df = handle_schema_drift(clean_df, config["silver_path"], table_name)
        
        # Step 5: Apply SCD or Simple Upsert
        print(f"\n[5/5] Writing to Silver layer...")
        
        if config["scd_type"] == 2:
            # Dimension table - apply SCD Type 2
            apply_scd_type2(
                clean_df, 
                config["silver_path"], 
                config["business_key"], 
                config["compare_columns"]
            )
        else:
            # Fact table - simple upsert
            simple_upsert(
                clean_df,
                config["silver_path"],
                config["business_key"]
            )
        
        print(f"\n✓ {table_name} processing COMPLETED successfully!")
        return True
        
    except Exception as e:
        print(f"\n✗ ERROR processing {table_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

In [0]:
# ============================================================
# CELL 9: Execute Processing for All Tables
# ============================================================

print("\n" + "="*60)
print("STARTING BRONZE TO SILVER TRANSFORMATION")
print("="*60)

results = {}

for config in table_configs:
    success = process_table(config)
    results[config["name"]] = "SUCCESS" if success else "FAILED"


STARTING BRONZE TO SILVER TRANSFORMATION

PROCESSING: CUSTOMERS

[1/5] Reading from Bronze: abfss://bronze@storagefordeproject1.dfs.core.windows.net/customers/2026/03/01/
✓ Read 1000 records from Bronze

[2/5] Running data quality checks...

--- Data Quality Checks: customers ---
Initial record count: 1000
After deduplication: 1000 (removed 0)
After null check: 1000 (removed 0)
✓ Data quality checks passed: 1000 clean records

[3/5] Checking for late arrivals...

[4/5] Checking schema drift...

--- Checking Schema Drift ---
✓ No schema drift detected

[5/5] Writing to Silver layer...

--- Applying SCD Type 2 ---
✓ Target table exists, performing MERGE...
✓ Closed changed records

✗ ERROR processing customers: 'str' object is not callable

PROCESSING: PRODUCTS

[1/5] Reading from Bronze: abfss://bronze@storagefordeproject1.dfs.core.windows.net/products/2026/03/01/


Traceback (most recent call last):
  File "/home/spark-b372b605-176f-41a6-a2a9-5a/.ipykernel/2108/command-7665259010649402-3835201572", line 37, in process_table
    apply_scd_type2(
  File "/home/spark-b372b605-176f-41a6-a2a9-5a/.ipykernel/2108/command-7665259010649397-3664291448", line 50, in apply_scd_type2
    .filter((col("end_date") == current_date()) & (col("is_current") == False))
             ^^^^^^^^^^^^^^^
TypeError: 'str' object is not callable


✓ Read 500 records from Bronze

[2/5] Running data quality checks...

--- Data Quality Checks: products ---
Initial record count: 500
After deduplication: 500 (removed 0)
After null check: 500 (removed 0)
✓ Data quality checks passed: 500 clean records

[3/5] Checking for late arrivals...

[4/5] Checking schema drift...

--- Checking Schema Drift ---
✓ No schema drift detected

[5/5] Writing to Silver layer...

--- Applying SCD Type 2 ---
✓ Target table exists, performing MERGE...
✓ Closed changed records

✗ ERROR processing products: 'str' object is not callable

PROCESSING: ORDERS

[1/5] Reading from Bronze: abfss://bronze@storagefordeproject1.dfs.core.windows.net/orders/2026/03/01/


Traceback (most recent call last):
  File "/home/spark-b372b605-176f-41a6-a2a9-5a/.ipykernel/2108/command-7665259010649402-3835201572", line 37, in process_table
    apply_scd_type2(
  File "/home/spark-b372b605-176f-41a6-a2a9-5a/.ipykernel/2108/command-7665259010649397-3664291448", line 50, in apply_scd_type2
    .filter((col("end_date") == current_date()) & (col("is_current") == False))
             ^^^^^^^^^^^^^^^
TypeError: 'str' object is not callable


✓ Read 10000 records from Bronze

[2/5] Running data quality checks...

--- Data Quality Checks: orders ---
Initial record count: 10000
After deduplication: 5000 (removed 5000)
After null check: 5000 (removed 0)
✓ Data quality checks passed: 5000 clean records

[3/5] Checking for late arrivals...

--- Checking for Late Arrivals ---
✓ Found 0 late arrival records

[4/5] Checking schema drift...

--- Checking Schema Drift ---
✓ No schema drift detected

[5/5] Writing to Silver layer...

--- Performing Simple Upsert ---
✓ Target exists, performing MERGE...
✓ Upsert completed

✓ orders processing COMPLETED successfully!

PROCESSING: ORDER_ITEMS

[1/5] Reading from Bronze: abfss://bronze@storagefordeproject1.dfs.core.windows.net/order_items/2026/03/01/
✓ Read 44694 records from Bronze

[2/5] Running data quality checks...

--- Data Quality Checks: order_items ---
Initial record count: 44694
After deduplication: 5000 (removed 39694)
After null check: 5000 (removed 0)
✓ Data quality checks pa

In [0]:
# ============================================================
# CELL 10: Summary Report
# ============================================================

print("\n" + "="*60)
print("TRANSFORMATION SUMMARY")
print("="*60)

for table_name, status in results.items():
    icon = "✓" if status == "SUCCESS" else "✗"
    print(f"{icon} {table_name}: {status}")

all_success = all(status == "SUCCESS" for status in results.values())

if all_success:
    print("\n✓ ALL TABLES PROCESSED SUCCESSFULLY!")
else:
    print("\n⚠ SOME TABLES FAILED - CHECK LOGS ABOVE")

print("="*60)


TRANSFORMATION SUMMARY
✗ customers: FAILED
✗ products: FAILED
✓ orders: SUCCESS
✓ order_items: SUCCESS

⚠ SOME TABLES FAILED - CHECK LOGS ABOVE
